# ReAct Agent from Scratch

## What is ReAct?

In large language models (LLMs), **(Re)asoning and (Act)ing (ReAct)** is a prominent framework that enables AI agents to perform complex, multi-step tasks by interleaving thoughts (internal reasoning) and actions (using external tools). This approach allows LLMs to dynamically interact with their environment and overcome limitations like factual hallucinations.

### **How ReAct Works: The Thought-Action-Observation Loop**

Inspired by human problem-solving, ReAct operates through a continuous feedback loop that involves three main components:

1.  **Thought**: The LLM generates a natural language reasoning step, breaking down the problem, assessing the current situation, and deciding the next logical step. This makes the agent's decision-making process transparent and debuggable.
2.  **Action**: Based on its thought, the model executes a predefined action using external tools or APIs. These tools could include web search engines, calculators, code interpreters, or internal databases.
3.  **Observation**: The model receives the result or feedback from the executed action. This new information updates the agent's context and informs its subsequent thoughts, allowing it to adapt its approach and self-correct as needed.

This cycle repeats until the agent determines it has enough information to provide a final, well-supported answer to the user's initial query.

### **Benefits of the ReAct Framework**

The ReAct framework offers several advantages over traditional LLM methods, such as standard Chain-of-Thought (CoT) prompting:

*   **Reduced Hallucination**: By retrieving and using real-time, external information, ReAct grounds the model's responses in facts, significantly reducing the generation of incorrect or outdated information.
*   **Enhanced Adaptability**: The dynamic, iterative nature allows ReAct agents to handle unforeseen obstacles and complex, unpredictable scenarios by adjusting their plan based on new observations.
*   **Improved Explainability**: The explicit thought traces provide insight into how the agent arrives at its conclusions, which facilitates debugging and increases user trust.
*   **Access to External Knowledge**: ReAct enables LLMs to interact with the real world beyond their training data, providing up-to-date and specific information.

## Step 1: Setup

First, install the OpenAI client and load the API key from Colab Secrets or an environment variable. The rest of the notebook focuses on the Thought -> Action -> Observation loop, not on key management.

In [ ]:
%%capture
!pip install -q openai python-dotenv
print('Done')


In [ ]:
import os
import re
from openai import OpenAI

def get_secret(name: str, *, required: bool = True):
    value = os.environ.get(name)
    try:
        from google.colab import userdata
        value = userdata.get(name) or value
    except Exception:
        pass
    if required and not value:
        raise ValueError(
            f"Missing {name}. In Colab, add a secret named {name} and enable Notebook access."
        )
    return value

api_key = get_secret('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)
print('OpenAI client ready')


Let's test the connection with a simple API call.

In [ ]:
chat_completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Hello world"}]
)
chat_completion.choices[0].message.content

## Step 2: The Agent Class

We will define a generic `Agent` class. This class will manage the conversation history (`messages`) and the interaction with the OpenAI API.

*   `__init__`: Initializes the agent with a `system` message (the instructions) and an empty list of messages.
*   `__call__`: Allows us to call the agent object like a function. It adds the user's message to the history, executes the LLM call, appends the assistant's response, and returns the result.
*   `execute`: Sends the current message history to the LLM and gets the response.

In [ ]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
                        model="gpt-4o-mini",
                        temperature=0,
                        messages=self.messages)
        return completion.choices[0].message.content

### Agent Interaction Example
Let's create a simple "helpful assistant" agent and see how it maintains history.

In [ ]:
agent = Agent('You are a helpful assistant')
agent("hello world")

In [ ]:
print(agent.messages)

## Step 3: Defining Tools

A dummy agent is not very useful if it can't do anything. For a ReAct agent, we need to give it "Tools" (Actions) it can perform.
Here we define two simple tools:
1.  `calculate`: A Python-based calculator.
2.  `average_dog_weight`: A mock database function that returns the weight of a dog breed.

In [ ]:
def calculate(what):
    # NOTE: eval() executes arbitrary Python code — safe here because the LLM
    # is generating arithmetic expressions, but NEVER use eval() on untrusted
    # user input in a production application. Use a proper math parser instead.
    return eval(what)

def average_dog_weight(name):
    if name in "Scottish Terrier":
        return("Scottish Terriers average 20 lbs")
    elif name in "Border Collie":
        return("a Border Collies average weight is 37 lbs")
    elif name in "Toy Poodle":
        return("a toy poodles average weight is 7 lbs")
    else:
        return("An average dog weights 50 lbs")

known_actions = {
    "calculate": calculate,
    "average_dog_weight": average_dog_weight
}

## Step 4: The ReAct Prompt

This is the core of the ReAct framework. We need to instruct the LLM on *how* to use its thoughts and actions.
The prompt defines a specific format:
*   **Thought**: The model reasons about what to do.
*   **Action**: The model picks a tool to run.
*   **PAUSE**: The model stops to let us run the tool.
*   **Observation**: We feed the tool's output back to the model.
*   **Answer**: The final response.

In [ ]:
prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

average_dog_weight:
e.g. average_dog_weight: Collie
returns average weight of a dog when given the breed

Example session:

Question: How much does a Bulldog weigh?
Thought: I should look the dogs weight using average_dog_weight
Action: average_dog_weight: Bulldog
PAUSE

You will be called again with this:

Observation: A Bulldog weights 51 lbs

You then output:

Answer: A bulldog weights 51 lbs
""".strip()

## Step 5: The Interaction Loop

Now we write the loop that automates this process.
1.  **Regex**: We use a regular expression to find lines starting with `Action:`.
2.  **Query Loop**: We repeatedly call the agent.
    *   If the agent returns an `Action`, we execute it.
    *   We append the `Observation` (result) to the prompt.
    *   We call the agent again with the new observation.
    *   This continues until the agent provides an `Answer` or we hit a max turn limit.

In [ ]:
action_re = re.compile('^Action: (\w+): (.*)$')   # python regular expression to selection action

In [ ]:
def query(question, max_turns=5):
    i = 0
    bot = Agent(prompt)
    next_prompt = question
    while i < max_turns:
        i += 1
        result = bot(next_prompt)
        print(result)
        actions = [
            action_re.match(a)
            for a in result.split('\n')
            if action_re.match(a)
        ]
        if actions:
            # There is an action to run
            action, action_input = actions[0].groups()
            if action not in known_actions:
                raise Exception("Unknown action: {}: {}".format(action, action_input))
            print(" -- running {} {}".format(action, action_input))
            observation = known_actions[action](action_input)
            print("Observation:", observation)
            next_prompt = "Observation: {}".format(observation)
        else:
            return

### Putting it all together
Let's ask a complex question that requires multiple tools: "I have 2 dogs, a border collie and a scottish terrier. What is their combined weight?"
The model should:
1.  Look up the weight of the Border Collie.
2.  Look up the weight of the Scottish Terrier.
3.  Calculate the sum.

In [ ]:
question = """I have 2 dogs, a border collie and a scottish terrier. \
What is their combined weight"""
query(question)

---

## What's Next: Beyond Custom Agents — The MCP Standard

You just built a ReAct agent from scratch. That gives you a deep understanding of *how* agentic loops work. In practice, the AI industry is converging on a standard way to make tools and context available to agents — without every developer having to re-implement the plumbing themselves.

### The Model Context Protocol (MCP)

**MCP (Model Context Protocol)** is an open standard released by Anthropic in 2024 and now supported by OpenAI, Google, and most major AI tooling vendors. Think of it as the **HTTP of AI tool integration** — a universal protocol for connecting AI models to external data sources and tools.

#### The Problem MCP Solves

Before MCP, every AI application had to write its own custom integration for every tool:
- Your agent needs to query a database → write custom Python function
- Another team's agent needs the same database → they write their own function too
- An LLM provider changes their function-calling API → every integration breaks

MCP standardizes this, so a tool is built *once* as an MCP server and can be used by *any* MCP-compatible agent or LLM.

#### How MCP Fits into What You've Learned

```
What you built today (ReAct from scratch):
  LLM ←→ Your custom tool functions (Python)

What LangChain does (abstraction layer):
  LLM ←→ LangChain Tool objects ←→ Your code

What MCP does (standardized protocol):
  LLM / Agent ←→ MCP Client ←→ MCP Server ←→ Database / API / Filesystem
```

MCP is especially powerful for **Agentic RAG** and multi-agent systems (covered in Module 7), where different agents may need to share the same tools.

#### MCP Architecture at a Glance

| Component | Role | Example |
|-----------|------|---------|
| **MCP Host** | The application running the AI model | Your Python app or Claude Desktop |
| **MCP Client** | Inside the host; speaks the MCP protocol | LangChain MCP adapter, OpenAI tools |
| **MCP Server** | Exposes tools, resources, and prompts | A node.js server wrapping your DB |
| **Transport** | How client ↔ server communicate | stdio (local), HTTP+SSE (remote) |

#### MCP vs. Function Calling

You used function calling earlier in this module. Here's how it compares:

| | Function Calling | MCP |
|---|---|---|
| **Scope** | Single model + your code | Any model + any MCP server |
| **Portability** | Specific to vendor's API | Vendor-agnostic standard |
| **Tool sharing** | Not standardized | Built-in — servers are reusable |
| **Use case** | Quick integrations | Production, multi-agent systems |

> **MCP is not replacing function calling** — it standardizes how tools are exposed and discovered, while function calling is still the mechanism the model uses to invoke them.

#### Quick Example: What an MCP Server Exposes

```python
# An MCP server might expose tools like:
{
    "tools": [
        {
            "name": "query_database",
            "description": "Run a read-only SQL query against the product database",
            "inputSchema": {
                "type": "object",
                "properties": {
                    "sql": {"type": "string", "description": "The SQL query to run"}
                }
            }
        },
        {
            "name": "search_documents", 
            "description": "Semantic search over internal documents",
            "inputSchema": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "top_k": {"type": "integer", "default": 5}
                }
            }
        }
    ]
}
# Any MCP-compatible agent can now discover and use these tools
# without knowing anything about the underlying implementation
```

#### Where to Learn More
- [MCP Specification](https://spec.modelcontextprotocol.io/)
- [Anthropic MCP Quickstart](https://modelcontextprotocol.io/quickstart)
- [LangChain MCP Integration](https://python.langchain.com/docs/integrations/toolkits/mcp/)
- MCP servers for popular tools (filesystem, GitHub, Slack, databases) are available at: [github.com/modelcontextprotocol/servers](https://github.com/modelcontextprotocol/servers)


---

## 🎯 Module Summary

In this module, you progressed from writing single prompts to building full autonomous agents:

1. **Prompting Techniques** — Zero-shot, few-shot, Chain-of-Thought, Self-Consistency, Tree of Thoughts, Prompt Chaining
2. **Responsible Prompting** — Prompt injection, leaking, hijacking, and mitigation strategies  
3. **Function Calling (Manual)** — Understanding the raw API: model requests a tool → you run it → model gets the result
4. **Function Calling (LangChain)** — Abstracting the loop with `create_tool_calling_agent` and `AgentExecutor`
5. **ReAct Agents** — The Thought/Action/Observation loop that powers modern AI agents
6. **MCP Standard** — How the industry is converging on a universal protocol for tool and context integration

**Next: Module 7 — Retrieval-Augmented Generation (RAG)**  
You'll apply these agentic concepts to build AI systems that can retrieve relevant documents in real-time and answer questions grounded in your own data.
